# Unit 12 — Noise-aware residual blocks

A denoiser needs both noise level and observation state. Sinusoidal timestep codes and FiLM-conditioned residual blocks carry those values through the network.

**Your result:** the two blocks that tell a denoiser *how noisy its input is* and *what the robot sees*. Your versions will match the pinned LeRobot diffusion-policy blocks, for outputs and gradients, including the weights of the prepared checkpoint.

Full answers are in [the reference notebook](12-diffusion-blocks-reference.ipynb).

**Sources.** [Diffusion Policy](https://diffusion-policy.cs.columbia.edu/) (1D temporal U-Net with FiLM conditioning), FiLM: [Perez et al. 2018](https://arxiv.org/abs/1709.07871), and the timestep embedding from [DDPM](https://arxiv.org/abs/2006.11239) (appendix B, after Vaswani et al.). Pinned code: `DiffusionSinusoidalPosEmb`, `DiffusionConv1dBlock` and `DiffusionConditionalResidualBlock1d` in `lerobot/policies/diffusion/modeling_diffusion.py` (revision `e0d50211ef236143ae867228662b7dfaba554f02`).

**Where this unit sits.** In Unit 11 the denoiser was supplied. Here you open it. Unit 13 assembles these blocks into the full U-Net. Your Unit 11 functions are available as package versions; this unit's functions are wired into the real network through hooks.


In [ ]:
import math
import sys
from pathlib import Path

import torch
from torch import nn

WORKSPACE = next(
    p
    for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "manifests/act_references.yaml").is_file()
)
sys.path.insert(0, str(WORKSPACE / "src"))
from lerobot.policies.diffusion.modeling_diffusion import (
    DiffusionConditionalResidualBlock1d,
    DiffusionSinusoidalPosEmb,
)

from vla_lab.teaching.blocks import FilmParts
from vla_lab.teaching.diffusion import ConditionalResidualBlock1d

In [ ]:
# Supplied: a helper that the checks use
def close(actual: torch.Tensor, expected: torch.Tensor, message: str, atol: float = 1e-5) -> None:
    """Compare two tensors and say what failed."""
    assert actual.shape == expected.shape, (
        f"{message}: shape {tuple(actual.shape)}, expected {tuple(expected.shape)}"
    )
    worst = (actual.detach() - expected.detach()).abs().max().item()
    assert worst <= atol, f"{message}: largest difference {worst:.2e} is above {atol:.0e}"

## Background: how a convolutional denoiser gets its conditions

The denoiser is a 1D convolutional network over the *time axis of the action window* (16 steps). It receives two kinds of side information:

1. **How noisy is the input?** The integer timestep `t` is turned into a smooth vector (`sinusoidal_timestep_embedding`) and passed through a small MLP.
2. **What does the robot see?** The observation features (state, camera features) form one *global condition* vector.

The two are concatenated into a `cond` vector. Every residual block turns `cond` into per-channel numbers that *modulate* the convolution features. This is **FiLM** (feature-wise linear modulation): `features = scale * features + bias`, with `scale` and `bias` computed from `cond`. In the pinned policy `use_film_scale_modulation=True`, so both scale and bias are used.
### Worked example: FiLM on two channels

Suppose one time slot has convolution features `[1, 1]`, FiLM scale `[2, 0.5]`, and bias `[-1, 1]`. Element-wise `scale * feature + bias` gives `[1, 1.5]`. The same channel values broadcast across all action-window slots. For embedding dimension 4 at timestep 0, the sine half is `[0, 0]` and cosine half is `[1, 1]`; a learned MLP then turns this code into the scale and bias.


## Exercise 12.1 — Sinusoidal timestep embedding

**Purpose.** Give the network a smooth, bounded code for the noise level, instead of a raw integer.

**Contract (shapes, units, masks).**

- `sinusoidal_timestep_embedding(timesteps, dim)`: `timesteps (B,)` integer or float tensor; `dim` even and at least 4. Return `(B, dim)` `float32`.
- `half = dim // 2`. Frequencies: `f_k = exp(-k * log(10000) / (half - 1))` for `k = 0 … half-1`.
- Output row = `[sin(t * f_0) … sin(t * f_{half-1}), cos(t * f_0) … cos(t * f_{half-1})]`: **all sines first, then all cosines** (not interleaved).
- Timestep 0 gives `sin = 0` and `cos = 1` everywhere.

**Method and source.** The Transformer position code applied to the diffusion timestep ([DDPM](https://arxiv.org/abs/2006.11239), appendix B; [Vaswani et al. 2017](https://arxiv.org/abs/1706.03762), 3.5). The layout differs from Exercise 7.4 (that one interleaves sin and cos and counts frequencies differently).

**Pointers.** Make the frequency vector first. Multiply the column of timesteps with the frequency row to get `(B, half)` angles. Concatenate the sines and the cosines on the last axis.

<details><summary>Detailed hint</summary>

```python
half = dim // 2
scale = math.log(10000) / (half - 1)
frequencies = torch.exp(torch.arange(half, device=timesteps.device) * -scale)
angles = timesteps.float().unsqueeze(-1) * frequencies.unsqueeze(0)
return torch.cat((angles.sin(), angles.cos()), dim=-1)
```

</details>

*This is the reference notebook: the next cell is a complete solution.*

In [ ]:
# Reference solution for Exercise 12.1
def sinusoidal_timestep_embedding(timesteps: torch.Tensor, dim: int) -> torch.Tensor:
    """Embed integer timesteps as sin/cos features, shape ``(B, dim)`` (Vaswani 3.5, DDPM B).

    ``half = dim // 2``; frequency ``f_k = exp(-k * log(10000) / (half - 1))`` for ``k = 0 .. half - 1``;
    output ``[sin(t * f), cos(t * f)]`` (all sines first, then all cosines).
    """
    half = dim // 2
    scale = math.log(10000) / (half - 1)
    frequencies = torch.exp(torch.arange(half, device=timesteps.device) * -scale)
    angles = timesteps.float().unsqueeze(-1) * frequencies.unsqueeze(0)
    return torch.cat((angles.sin(), angles.cos()), dim=-1)

In [ ]:
# Check: Exercise 12.1
t = torch.tensor([0, 1, 7, 99])
for dim in (16, 32, 128):
    mine = sinusoidal_timestep_embedding(t, dim)
    assert mine.shape == (4, dim) and mine.dtype == torch.float32, f"Expected float32 (4, {dim})."
    close(
        mine,
        DiffusionSinusoidalPosEmb(dim)(t),
        f"dim {dim} differs from the pinned embedding",
        atol=1e-6,
    )
zero = sinusoidal_timestep_embedding(torch.tensor([0]), 8)
assert torch.equal(zero[0, :4], torch.zeros(4)) and torch.equal(zero[0, 4:], torch.ones(4)), (
    "Timestep 0 is all sines 0 then all cosines 1."
)
close(
    sinusoidal_timestep_embedding(torch.tensor([5]), 8)[0, 0],
    torch.tensor(math.sin(5.0)),
    "first frequency is 1",
    atol=1e-6,
)
close(
    sinusoidal_timestep_embedding(torch.tensor([5]), 8)[0, 3],
    torch.tensor(math.sin(5 * 1e-4)),
    "last frequency is 1/10000",
    atol=1e-6,
)
print("Exercise 12.1 passed")

### What the residual path contributes

```
x ─────────────── identity or 1×1 convolution ───────┐
│                                                   + ── output
└─ conv1 → FiLM scale and bias → conv2 ─────────────┘
```

The main path learns a change to the input features. The shortcut carries the original features to the addition, which gives optimization a direct path through the block. When input and output channel counts match, the shortcut is the identity. When they differ, a learned 1×1 convolution changes the channel count. It preserves the time length. FiLM modulates the main path; it does not replace the shortcut.



## Exercise 12.2 — FiLM-conditioned residual block

**Purpose.** Write the block that is repeated through the whole U-Net: two convolutions, with the condition modulating the features between them.

**Contract (shapes, units, masks).**

- `block` has `conv1`, `conv2` (`Conv1dBlock`: Conv1d → GroupNorm → Mish, keeps the length), `cond_encoder` (`Mish → Linear(cond_dim, K)`), `residual_conv` (`Conv1d(in, out, 1)` or `Identity`), `out_channels` (`C`) and the bool `use_film_scale_modulation`.
- `x (B, C_in, T)` (channels first), `cond (B, cond_dim)`. Return `(B, C, T)`.
- `out = conv1(x)`. `embed = cond_encoder(cond)` reshaped to `(B, K, 1)` so it broadcasts over `T`.
- **With scale modulation** (`K = 2C`): `scale = embed[:, :C]`, `bias = embed[:, C:]`, `out = scale * out + bias`. **Without** (`K = C`): `out = out + embed`.
- `out = conv2(out)`. Return `out + residual_conv(x)`.
- The pinned code uses `scale * out + bias`, not `(1 + scale) * out + bias`.

**Method and source.** FiLM: `γ(c) ⊙ F + β(c)` — [Perez et al. 2018](https://arxiv.org/abs/1709.07871). Diffusion Policy uses it in each 1D residual block ([paper site](https://diffusion-policy.cs.columbia.edu/)). Pinned class: `DiffusionConditionalResidualBlock1d`.

**Pointers.** Follow the order of the contract line by line. The only decision is how to cut `embed` into `scale` and `bias`. The residual connection adds the *input* (after `residual_conv`), not the output of `conv1`.

<details><summary>Detailed hint</summary>

```python
out = block.conv1(x)
embed = block.cond_encoder(cond).unsqueeze(-1)              # (B, K, 1)
if block.use_film_scale_modulation:
    scale, bias = embed[:, : block.out_channels], embed[:, block.out_channels :]
    out = scale * out + bias
else:
    out = out + embed
out = block.conv2(out)
return out + block.residual_conv(x)
```

</details>

*This is the reference notebook: the next cell is a complete solution.*

In [ ]:
# Reference solution for Exercise 12.2
def film_residual_forward(block: FilmParts, x: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
    """FiLM-conditioned residual block (Perez et al. 2018, used in Diffusion Policy).

    ``x (B, C_in, T)``, ``cond (B, cond_dim)``, returns ``(B, C_out, T)``. Steps:
    ``out = conv1(x)``; ``e = cond_encoder(cond)`` reshaped to ``(B, K, 1)``. With scale
    modulation ``K = 2 * C_out``: ``scale = e[:, :C_out]``, ``bias = e[:, C_out:]`` and
    ``out = scale * out + bias``. Without it ``K = C_out`` and ``out = out + e``.
    Then ``out = conv2(out)`` and the result is ``out + residual_conv(x)``.
    Block attributes: ``conv1``, ``cond_encoder``, ``conv2``, ``residual_conv``,
    ``out_channels``, ``use_film_scale_modulation``.
    """
    out = block.conv1(x)
    embed = block.cond_encoder(cond).unsqueeze(-1)
    if block.use_film_scale_modulation:
        scale, bias = embed[:, : block.out_channels], embed[:, block.out_channels :]
        out = scale * out + bias
    else:
        out = out + embed
    out = block.conv2(out)
    return out + block.residual_conv(x)

In [ ]:
# Check: Exercise 12.2
for scale in (True, False):
    torch.manual_seed(2)
    oracle = DiffusionConditionalResidualBlock1d(
        8, 16, 10, kernel_size=5, n_groups=8, use_film_scale_modulation=scale
    )
    mine = ConditionalResidualBlock1d(8, 16, 10, 5, 8, scale, forward_fn=film_residual_forward)
    mine.load_state_dict(oracle.state_dict(), strict=True)  # same layer names: identical weights
    x, cond = torch.randn(3, 8, 16, requires_grad=True), torch.randn(3, 10, requires_grad=True)
    out = mine(x, cond)
    assert out.shape == (3, 16, 16), "Expected (B, out_channels, T)."
    close(out, oracle(x, cond), f"output (scale modulation={scale}) differs from the pinned block")
    out.square().sum().backward()
    x2, cond2 = (t.detach().clone().requires_grad_(True) for t in (x, cond))
    oracle(x2, cond2).square().sum().backward()
    close(x.grad, x2.grad, f"input gradient (scale={scale})", atol=1e-5)
    close(cond.grad, cond2.grad, f"condition gradient (scale={scale})", atol=1e-5)
    close(
        mine.cond_encoder[1].weight.grad,
        oracle.cond_encoder[1].weight.grad,
        "FiLM layer gradient",
        atol=1e-5,
    )
    assert not torch.allclose(mine(x, cond + 1.0), out.detach(), atol=1e-4), (
        "The condition must change the output."
    )
same = ConditionalResidualBlock1d(16, 16, 10, 5, 8, True, forward_fn=film_residual_forward)
assert isinstance(same.residual_conv, nn.Identity) and same(
    torch.randn(2, 16, 16), torch.randn(2, 10)
).shape == (2, 16, 16), "Also check equal channels."
print("Exercise 12.2 passed")